### Proje için gerekli kütüphaneleri import etme
Bu projemizde veri manipülasyonunda kullanılan NumPy ve Pandas kütüphanelerini kullanacağız.

Pandas kütüphanesinin fonksiyonlarına pd alias'ı ile erişeceğiz.
NumPy kütüphanesinin fonksiyonlarına np alias'ı ile erişeceğiz.

In [1]:
import pandas as pd
import numpy as np

### Veriyi CSV formatından DataFrame formatına dönüştürme

In [2]:
df = pd.read_csv("monthly_sales.csv")

### EDA

In [3]:
# Veri setinin satır ve sütun bilgisi ile veri setini keşfetmeye başlıyoruz.
print(f"Veri setindeki satır sayısı: {df.shape[0]}")
print(f"Veri setindeki sütun sayısı: {df.shape[1]}")

Veri setindeki satır sayısı: 24
Veri setindeki sütun sayısı: 4


In [4]:
# Kafamızda veri setine dair birşeylerin canlanması için veri setinin ilk 5 satırına bakalım.
print(df.head())

         date  revenue  orders    region
0  2023-01-01  11602.0   106.0  Istanbul
1  2023-02-01  13435.0   114.0    Ankara
2  2023-03-01  11360.0   140.0     İzmir
3  2023-04-01      NaN   137.0     İzmir
4  2023-05-01  15106.0   134.0  Istanbul


In [5]:
# veri setindeki sütunlar arasında boş değer var mı kontrol edelim.
print(df.isna().sum())

date       0
revenue    1
orders     1
region     0
dtype: int64


In [6]:
# veri setindeki sayısal sütunlara dair matematiksel işlemler yaparak EDA'ya devam ediyoruz.
print(f"veri setindeki matematiksel sütunlara dair işlemler:\n{df.describe()}")

veri setindeki matematiksel sütunlara dair işlemler:
            revenue      orders
count     23.000000   23.000000
mean   14884.086957  161.565217
std     5364.846172   38.531954
min    -5000.000000  106.000000
25%    13132.500000  134.000000
50%    15106.000000  148.000000
75%    17591.500000  182.000000
max    22843.000000  231.000000


In [7]:
# veri setine genel bir bakış atacağız. Bunun için Pandas kütüphanesinin info fonksiyonunu kullanacağız.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   date     24 non-null     str    
 1   revenue  23 non-null     float64
 2   orders   23 non-null     float64
 3   region   24 non-null     str    
dtypes: float64(2), str(2)
memory usage: 900.0 bytes


### VERİ TEMİZLEME

In [8]:
df["date"] = pd.to_datetime(df["date"]) # Zaman serisi analizleri için string tipindeki tarih sütununu datetime tipine çevirmeliyiz

In [9]:
## Revenue sütunu için outlier tespiti
q1 = df["revenue"].quantile(0.25)
q3 = df["revenue"].quantile(0.75)
iqr = q3-q1
alt_sinir=q1 - (1.5*iqr)
ust_sinir = q3 + (1.5*iqr)
print(f"revenue sütununda aykırı değere sahip satırlar:\n")
print(df[(df["revenue"]<alt_sinir) & (df["revenue"]>ust_sinir)])

revenue sütununda aykırı değere sahip satırlar:

Empty DataFrame
Columns: [date, revenue, orders, region]
Index: []


In [10]:
## Orders sütunu için outlier tespiti
q1= df["orders"].quantile(0.25)
q3 = df["orders"].quantile(0.75)
iqr = q3-q1
alt_sinir = q1 - (1.5*iqr)
ust_sinir = q3 + (1.5*iqr)
print(f"orders sütununda aykırı değere sahip satırlar:\n")
print(df[(df["orders"]<alt_sinir) | (df["orders"]>ust_sinir)])

orders sütununda aykırı değere sahip satırlar:

Empty DataFrame
Columns: [date, revenue, orders, region]
Index: []


Yapılan outlier tespiti sonuçlarına göre orders ve revenue sütunlarında aykırı değerler olmadığından dolayı bu sütunlardaki boş hücrelerin ilgili sütunun ortalama değeriyle doldurulmasına karar verilmiştir. Ayrıca revenue negatif olamayacağından dolayı revenue miktarı negatif olan satırlar veri setinden kaldırılacaktır.

In [11]:
# boş hücreleri dolduralım, yanlış veriyi silelim ve bu dataframe'i temizlenmiş bir dataframe olarak yeni bir değişkene atayalım.
df["revenue"] = df["revenue"].fillna(df["revenue"].mean())
df["orders"] = df["orders"].fillna(df["orders"].mean())
cleaned_df = df[df["revenue"]>0]
print(f"Temizlenmiş veri setinin ilk 5 satırı:\n{cleaned_df.head()}")
print(f"Temizlenen veri setinde boş değer var mı?:\n{cleaned_df.isna().sum()}")

Temizlenmiş veri setinin ilk 5 satırı:
        date       revenue  orders    region
0 2023-01-01  11602.000000   106.0  Istanbul
1 2023-02-01  13435.000000   114.0    Ankara
2 2023-03-01  11360.000000   140.0     İzmir
3 2023-04-01  14884.086957   137.0     İzmir
4 2023-05-01  15106.000000   134.0  Istanbul
Temizlenen veri setinde boş değer var mı?:
date       0
revenue    0
orders     0
region     0
dtype: int64


### Veri Görselleştirme

In [12]:
import plotly.express as px # veriyi görselleştirmek için plotly kütüphanesini kullanacağız.Bu kütüphanenin fonksiyonlarına px alis'ı ile ulaşacağız.

In [13]:
fig = px.line( # zaman serisi analizi yapacağız. x ekseninde tarih, y ekseninde revenue olacak şekilde bir çizgi grafiği oluşturuyoruz. Bu sayede tarihler boyunca revenue'daki değişimi görsel olarak çok net görüp yorumlayabileceğiz
    cleaned_df,
    x="date",
    y="revenue",
    title="Zaman Serisi Grafiği: Revenue"
    )

In [14]:
fig.show() # oluşturduğumuz grafiği görüntülemek için show fonksiyonunu kullanıyoruz.

In [15]:
cleaned_df["growth_rate"] = cleaned_df["revenue"].pct_change() * 100 # revenue sütunundaki büyüme oranını hesaplayarak yeni bir sütun oluşturuyoruz.
cleaned_df.head() 

,date,revenue,orders,region,growth_rate
0,2023-01-01,11602.000000,106.0,Istanbul,NaN
1,2023-02-01,13435.000000,114.0,Ankara,15.799000
2,2023-03-01,11360.000000,140.0,İzmir,-15.444734
3,2023-04-01,14884.086957,137.0,İzmir,31.021892
4,2023-05-01,15106.000000,134.0,Istanbul,1.490942


In [16]:
fig1=px.line( # büyüme oranı = (sonraki aydaki revenue - önceki aydaki revenue) / (önceki aydaki revenue) * 100 formülü ile hesaplanır. Bölme olan kısmı Pandas'ın pct_change fonksiyonu ile buluruz.
    cleaned_df, # amacımız tarihler boyunca revenue'daki büyüme oranını (pozitif-negatif) görselleştirerek veriye dair net yorumlarda bulunabilmek
    x="date",
    y="growth_rate",
    title="Zaman Serisi Grafiği: Growth Rate"
)

In [17]:
fig1.show()

### Revenue Özet
* Genel olarak satışların her iki yılda da ekim-aralık ve mart-ağustos ayları arasında  arttığı tespit edilmiştir ancak bu artış ekim-aralık ayları arasında daha fazladır.
* En iyi dönem 2024 yılının Aralık ayıdır ve bu ayda 22.843 K gelir elde edilmiştir.
* En kötü dönem 2023 yılının Mart ayıdır ve bu ayda 11.36 K gelir elde edilmiştir.

### Growth Rate Özet
* En yüksek büyüme 2023 yılının Aralık ayındadır, bu ayda önceki aya kıyasla büyüme oranı %72 oranında artmıştır.
* En düşük büyüme ise 2024 yılının Ocak ayındadır, bu ayda önceki aya kıyasla büyüme oranı %40 azalmıştır.

### Şirkete Önerim
Ürünlerin stokları black friday gibi satışların zirve yaptığı dönemlerde tükenmemelidir. Bu dönemlerde, reklam kampanyalarına harcanan bütçe artırılabilir. Öte yandan satışın düşük olduğu dönemlerde teknolojik ürünlerde örneğin laptop alana mouse hediye gibi kampanyalar yapılabilir veya telefon alana kablosuz kulaklık hediye edilebilir.
